# 🚀 Phase 0: System-Setup, Hardware-Verification & Phasen-Steuerung

Dieses Notebook orchestriert das Gesamtsystem:
1. **Umgebung & Auth:** Setzt den NGC Key und meldet Docker an der Registry an.
2. **Hardware Check:** Garantiert GPU-Treiber & Auslastung.
3. **Phasen-Steuerung:** Schaltet dynamisch zwischen **Phase 1 (Großes SDG-LLM)** und **Phase 2 (Kleines Mistral-LLM + NeMo Customizer)** um.

## 1. Environment & Auth Setup

In [1]:
import sys
import os

# Projekt-Root verfügbar machen (für Import aus src/)
sys.path.append(os.path.abspath(".."))

from src.utils.setup import setup_environment, login_docker_registry, run_hardware_check

# 1. Environment & API Key prüfen/setzen
setup_environment()

# 2. Docker Login ausführen (ausgelagerte Logik mit Exception Handling)
login_docker_registry()

ValueError: ❌ NGC_API_KEY fehlt! Bitte in der .env-Datei eintragen oder als Parameter übergeben.

## 2. Hardware & GPU Check

In [2]:
# Validiert VRAM, NVIDIA Driver & Cuda Durchreichung
run_hardware_check("../check_hardware.py")


🖥️ Starte Hardware-Check...
🚀 STARTE HARDWARE- UND ENVIRONMENT-CHECK VOR DEM CONTAINER-START

🔍 [1/4] Prüfe freien Festplattenspeicher...
   --> Freier Speicherplatz: 7.90 GB

❌ HARDWARE CHECK FEHLGESCHLAGEN!
------------------------------------------------------------------
Fehlerdetails: Zu wenig Speicherplatz! Benötigt: mind. 250 GB, Verfügbar: 7.90 GB.
Tipp: Das Mistral-128B Modell und das NeMo Container-Image benötigen viel Platz.
------------------------------------------------------------------
Bitte behebe das obige Problem, bevor du 'docker compose up' startest.


RuntimeError: ❌ Hardware-Check mit Fehlern abgebrochen.

---
## 3. Phase 1: Datengenerierung starten (Großes SDG-LLM)

Startet ausschließlich das große LLM für die synthetische Datengenerierung (SDG) und den NeMo Data Designer.

In [ ]:
# Phase 1 Container im Hintergrund hochfahren
!docker compose --profile sdg up -d

In [ ]:
# Status prüfen & Logs des großen NIMs verfolgen
!docker compose ps
# Zeigt die letzten 10 Zeilen Logs an, um zu prüfen ob das Modell im VRAM lauft
!docker compose logs --tail=10 nim_large_sdg

> ➡️ **Nächster Schritt:** Wechsle jetzt zu **`01_data_generation.ipynb`**, um deine Trainingsdaten zu erzeugen.
>
> ⚠️ **Sobald die Datenerzeugung abgeschlossen ist**, führe die Zelle unten aus!

---

## 4. Phase 1 beenden & VRAM freigeben

Fährt die SDG-Container herunter, um den gesamten GPU-Speicher für das Fine-Tuning freizuräumen.

In [ ]:
# Stoppt Phase 1 Container und leert VRAM
!docker compose --profile sdg down
print("🧹 Phase 1 beendet. GPU VRAM ist wieder vollständig frei!")

---
## 5. Phase 2: Fine-Tuning & Curating starten (Kleines Mistral-LLM)

Startet den **NeMo Customizer** und das **kleine Mistral-7B NIM** für Data Cleaning, SFT-Training und Evaluation.

In [ ]:
# Container für Phase 2 starten
!docker compose --profile fine_tuning up -d
!docker compose ps

In [ ]:
# System Health Check für Endpunkte der Phase 2 ausführen
!python ../smoke_test.py